# Knowledge-to-Equation — Detailed, Self-Contained Walkthrough

This notebook is the **explanatory edition**. Unlike `playground.ipynb` (which just
calls the `src/` package), here the **core logic is written inline**, cell by cell,
with a detailed explanation of *what* each piece does and *why* it is built that way.

Read it top to bottom to understand the full method; run it cell by cell to watch the
pipeline build an equation for your process.

---

## The big idea

Classic process modelling needs either (a) a physics-based derivation, which is slow and
often intractable for novel processes, or (b) a lot of experimental data to fit a black-box
model. This framework takes a **third route**:

1. **Read the literature** with an LLM and *retrieve* the qualitative relationships
   between the process parameters and the performance metric (e.g. *'higher pulse energy
   increases material removal rate'*).
2. **Ask the LLM to turn that knowledge into a candidate equation** — a small Python
   function with free parameters.
3. **Fit** those free parameters to a *small* experimental dataset and **score** the fit.
4. **Refine** the best equations by feeding the LLM the scores and asking for improvements.

The payoff measured in the paper: equations that **extrapolate** well — they stay accurate
in a region of the parameter space that was never used for training.

### Pipeline at a glance
```
 literature PDFs --> [RAG retrieval] --+
                                       +--> [Phase 1: generate + fit candidate eqns]
 dataset.xlsx ------> [train/val/test]-+                 |
                                                          v
                                       [Phase 3: refine until validation R2 target]
                                                          |
                                                          v
                                       initial_equation.txt / final_equation.txt
```


## 0. Setup

`nest_asyncio` lets LlamaParse / LlamaIndex run their internal `asyncio` loops inside the
notebook's already-running loop. We then load the configuration and the API keys.

All process-specific choices (variable names, columns, file paths, iteration counts) live
in `config/config.yaml` — so the *same logic below* works for any process. We use the
project's tiny `config_loader`, then immediately unpack the few values we care about into
plain variables so the rest of the notebook reads clearly.


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from src.config_loader import load_config, require_env, resolve_path, variable_phrase

cfg = load_config()                 # reads config/config.yaml + loads config/.env keys

INPUTS       = cfg['variables']['inputs']      # ordered list of input variable dicts
OUTPUT       = cfg['variables']['output']      # the target metric dict
INPUT_SYMS   = cfg['input_symbols']            # e.g. ['WS', 'P', 'F', 'SS']
INPUT_COLS   = cfg['input_columns']            # matching spreadsheet column names
OUT_SYM      = cfg['output_symbol']            # e.g. 'MRR'
OUT_COL      = cfg['output_column']

print('Topic      :', cfg['topic'])
print('Inputs     :', INPUT_SYMS)
print('  columns  :', INPUT_COLS)
print('Output     :', OUT_SYM, '->', OUT_COL)


## 1. Knowledge extraction (Retrieval-Augmented Generation)

**Goal:** convert a pile of PDFs into a short block of text that states how each input
variable affects the output, grounded in the papers.

**How it works, step by step:**

1. **Parse** each PDF with *LlamaParse* into Markdown. PDFs are messy (multi-column, tables,
   equations); LlamaParse with a custom `parsing_instruction` produces clean Markdown that
   preserves tables and equations. Parsing is the slow/expensive part, so the result is
   **pickled to disk** and reused on later runs.
2. **Index** the parsed documents. `MarkdownElementNodeParser` splits the Markdown into
   nodes (paragraphs, tables, ...). We embed them with an OpenAI embedding model and store
   them in an in-memory `VectorStoreIndex` for semantic search.
3. **Retrieve + rerank.** A query engine pulls the `similarity_top_k` most similar nodes,
   then a **BGE reranker** (a cross-encoder) reorders them by true relevance — rerankers
   are far more accurate than raw embedding similarity for picking the best passages.
4. **Query.** We send one carefully-worded question asking specifically about the
   *direction and shape* (positive/negative, linear/log/exponential) of each
   input-to-output relationship. The answer text is what the equation-generator will read.

The function below is exactly `src/knowledge.py`, inlined so you can see every step.


In [ ]:
import os, pickle

def build_retriever(cfg):
    from llama_parse import LlamaParse
    from llama_index.core import Settings, VectorStoreIndex
    from llama_index.core.node_parser import MarkdownElementNodeParser
    from llama_index.embeddings.openai import OpenAIEmbedding
    from llama_index.llms.openai import OpenAI
    from llama_index.postprocessor.flag_embedding_reranker import FlagEmbeddingReranker

    kcfg, mcfg = cfg['knowledge'], cfg['model']
    openai_key = require_env('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = openai_key

    # --- (1) Parse PDFs (or load the cached parse) -------------------------------
    cache_path = resolve_path(cfg, kcfg['parsed_cache'])
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            documents = pickle.load(f)
    else:
        files = [str(resolve_path(cfg, p)) for p in kcfg['pdf_files']]
        parser = LlamaParse(api_key=require_env('LLAMA_CLOUD_API_KEY'),
                            result_type='markdown',
                            parsing_instruction=kcfg['parsing_instruction'])
        documents = parser.load_data(files)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        with open(cache_path, 'wb') as f:
            pickle.dump(documents, f)

    # --- (2) Embeddings + LLM used by the index ----------------------------------
    Settings.embed_model = OpenAIEmbedding(model=mcfg['embedding_model'], api_key=openai_key)
    Settings.llm = OpenAI(model=mcfg['llm_model'], api_key=openai_key)

    # --- (3) Build index over markdown nodes -------------------------------------
    node_parser = MarkdownElementNodeParser(llm=Settings.llm, num_workers=8)
    nodes = node_parser.get_nodes_from_documents(documents, progress=False)
    base_nodes, objects = node_parser.get_nodes_and_objects(nodes)
    index = VectorStoreIndex(nodes=base_nodes + objects)

    # --- (4) Retriever + reranker + the query engine -----------------------------
    reranker = FlagEmbeddingReranker(top_n=kcfg['reranker_top_n'], model=kcfg['reranker_model'])
    query_engine = index.as_query_engine(similarity_top_k=kcfg['similarity_top_k'],
                                          node_postprocessors=[reranker], verbose=False)

    # The retrieval query: {topic}/{inputs}/{output} are filled from the config.
    query = kcfg['retrieval_query'].format(
        topic=cfg['topic'],
        inputs=variable_phrase(INPUTS),
        output=f"{OUTPUT['name']} ({OUT_SYM})")

    def retrieve():
        return query_engine.query(query).response + '\n\n'
    return retrieve


Build the retriever and run one retrieval so you can **read the knowledge the LLM will use**.
Note `retrieve()` is a *callable*: every time it is called it re-runs the query. The phases
below call it on each iteration, so each candidate equation is grounded in a fresh retrieval.


In [ ]:
retrieve = build_retriever(cfg)
knowledge = retrieve()
print(knowledge)


## 2. The dataset and the *extrapolative* split

This is the heart of why the method is interesting. A normal random train/test split tests
**interpolation** (test points sit among the training points). Here we deliberately test
**extrapolation**:

- For each chosen `filter_columns`, we iteratively keep only rows **below** the `ratio`
  quantile. Repeating this across columns carves out the **low-value corner** of the
  parameter space. That corner becomes the train/validation pool.
- Everything *outside* that corner (the higher-value region the model never saw) becomes the
  **test set**. Good test scores therefore prove the equation generalises beyond the training
  region — exactly what a physically meaningful equation should do.
- From the low-value pool we randomly `sample` `train_size` rows for **training**; the rest
  is **validation** (used to rank candidate equations during the search).

Each split is returned as `(X, y)` where `X` is a **tuple of arrays, one per input variable,
in config order**. That tuple is exactly what the generated `model(X, ...)` functions unpack,
and what `scipy.optimize.curve_fit` expects as its `xdata`.


In [ ]:
import pandas as pd

def load_and_split(cfg):
    ds = cfg['dataset']
    df = pd.read_excel(resolve_path(cfg, ds['file']))

    # Carve out the low-value region: iteratively keep rows below the ratio quantile.
    filter_cols = ds.get('filter_columns') or INPUT_COLS
    pool = df.copy()
    for col in filter_cols:
        threshold = pool[col].quantile(ds['ratio'])
        pool = pool[pool[col] < threshold].copy()

    test_df = df.drop(pool.index)                                   # high-value -> test
    train_df = pool.sample(n=ds['train_size'], random_state=ds['random_state'])
    val_df = pool.drop(train_df.index)                              # remaining pool -> val

    def extract(d):
        X = tuple(d[c].values for c in INPUT_COLS)   # tuple of arrays, config order
        y = d[OUT_COL].values
        return X, y

    return {'train': extract(train_df), 'validation': extract(val_df),
            'test': extract(test_df), 'train_df': train_df}

data = load_and_split(cfg)
print('Train:', len(data['train'][1]),
      '| Validation:', len(data['validation'][1]),
      '| Test:', len(data['test'][1]))
data['train_df'].head()


## 3. Turning LLM text into a *fitted, scored* model

The LLM returns a **string** of Python source. We need to (a) safely turn it into a callable,
(b) fit its free parameters, and (c) score it. Four small helpers do this:

- **`clean_code`** strips the ```` ```python ```` fences the model often adds.
- **`extract_function`** `exec`s the code in an isolated namespace (with `np`/`math`
  available) and returns the first function it defines — found by regex on `def name(`.
  *Why detect the name instead of hard-coding `model`?* Phase 3 mixes functions named
  `model` and `improved_model`; detecting the name makes the same evaluator work for both.
- **`count_inputs`** reads the function signature to learn how many free parameters
  (`a0, a1, ...`) it has, so we know how many initial guesses to pass.
- **`fit_and_evaluate`** calls `curve_fit` on the **training** data (the parameters are
  optimised by non-linear least squares) and then reports **validation** MSE and R2 — the
  scores used to rank candidates. A bad equation (non-convergent, NaNs, wrong arity) raises,
  and the calling loop simply skips it.

> **Security note:** `exec` runs model-generated code. That is acceptable here because the
> source is your own LLM in a research setting, but never point this at untrusted input.


In [ ]:
import inspect, math, re
import numpy as np
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, r2_score

_EXEC_GLOBALS = {'np': np, 'numpy': np, 'math': math}

def clean_code(code):
    return code.replace('```python', '').replace('```', '')

def extract_function(code):
    m = re.search(r'def\s+(\w+)\s*\(', code)
    if not m:
        raise ValueError('No function definition found in generated code.')
    ns = dict(_EXEC_GLOBALS)
    exec(code, ns)                       # define the function in an isolated namespace
    return ns[m.group(1)]

def count_inputs(func):
    return len(inspect.signature(func).parameters)

def fit_and_evaluate(func, data):
    X_train, y_train = data['train']
    X_val,   y_val   = data['validation']
    n_params = count_inputs(func) - 1     # first parameter is X; the rest are a0, a1, ...
    popt, _ = curve_fit(func, X_train, y_train, p0=[1] * n_params)
    pred_val = func(X_val, *popt)
    return popt, mean_squared_error(y_val, pred_val), r2_score(y_val, pred_val)


## 4. The prompts (knowledge -> code)

Prompt engineering *is* the algorithm here, so it is worth reading closely. Two prompts:

**Initial prompt (Phase 1).** Gives the LLM the retrieved knowledge and asks for a function
in a *fixed skeleton* so the output is always parseable and fittable:
```python
def model(X, a0, ...):
    WS, P, F, SS = X      # symbols + order come straight from the config
    MRR = ...
    return MRR
```
It explicitly allows linear / exponential / logarithmic / trigonometric forms and says
'only use the retrieved information', keeping the equation grounded in the literature.

**Refinement prompt (Phase 3).** Adds the *summary of the current best models with their
scores* and asks the LLM to **modify** them (combine terms, swap operations, add terms) to
raise R2. This is the feedback loop that lets the search climb toward a better fit.

Both prompts are generated from the config, so the variable names, symbols and unpack order
always match the dataset.


In [ ]:
def initial_prompt(context):
    unpack = ', '.join(INPUT_SYMS)
    return (
        f"Use the following retrieved information:\n{context}\n"
        f"Based on the retrieved relationships between {variable_phrase(INPUTS)} "
        f"affecting {OUTPUT['name']} ({OUT_SYM}), generate a Python function to model "
        f"{OUT_SYM} as a function of the inputs: ({OUT_SYM}=f({unpack})). "
        "The function can be in various formats, including linear, exponential, "
        "logarithmic, or trigonometric functions. The function should be formatted as:\n\n"
        "def model(X, a0, ...):\n"
        f"    {unpack} = X\n"
        f"    {OUT_SYM} = ...\n"
        f"    return {OUT_SYM}\n\n"
        "Provide no other context or explanation, just the Python function code. "
        "No extra text. Only utilize the retrieved information to generate the equation.")

def refine_prompt(context, summary):
    unpack = ', '.join(INPUT_SYMS)
    return (
        f"Use the following retrieved information:\n{context}\n\n"
        f"The results for the top models were as follows:\n{summary}\n\n"
        "The R2 scores indicate how well each model fits the data. Models with higher R2 "
        "and lower MSE are more reliable and should be given more consideration.\n\n"
        f"Based on this information, generate an improved, complete function leading to a "
        f"better R2 for modelling {OUTPUT['name']} ({OUT_SYM}) as a function of "
        f"{variable_phrase(INPUTS)}. Use this format:\n\n"
        "def improved_model(X, a0, ...):\n"
        f"    {unpack} = X\n"
        f"    {OUT_SYM} = ...\n"
        f"    return {OUT_SYM}\n\n"
        "Alter the equations by combining or modifying operations (e.g. changing divisions "
        "to multiplications, combining or introducing terms). Do not add comments. Only "
        f"return complete Python code, ending with 'return {OUT_SYM}'.")


### The LLM calls

Two thin wrappers around the Chat Completions API. The Phase 1 call uses a **random
temperature** in a configured range — sampling at different temperatures diversifies the
candidate equations so the search explores more functional forms. The refinement call is
more focused. `max_tokens` is kept small because we only want a short function back.


In [ ]:
import random, openai

client = openai.OpenAI(api_key=require_env('OPENAI_API_KEY'))
mcfg = cfg['model']

def generate_initial(context):
    temperature = random.uniform(mcfg['temperature_min'], mcfg['temperature_max'])
    resp = client.chat.completions.create(
        model=mcfg['llm_model'],
        messages=[{'role': 'system', 'content': 'Generates Python code.'},
                  {'role': 'user',   'content': initial_prompt(context)}],
        max_tokens=mcfg['max_tokens_initial'], temperature=temperature)
    return resp.choices[0].message.content

def generate_refined(context, summary):
    resp = client.chat.completions.create(
        model=mcfg['llm_model'],
        messages=[{'role': 'system',
                   'content': f'Generates Python code. The function must always conclude with return {OUT_SYM}'},
                  {'role': 'user', 'content': refine_prompt(context, summary)}],
        max_tokens=mcfg['max_tokens_refine'])
    return resp.choices[0].message.content


## 5. Phase 1 — generate a population of equations

The search loop. On each iteration we: retrieve knowledge -> ask the LLM for an equation ->
clean it -> turn it into a function -> fit + score it. Anything that fails (unparseable,
non-convergent, produces NaNs) is caught and skipped without consuming an iteration, so we
always end up with `phase1_iterations` *successful* candidates.

We collect every successful candidate, then **sort by validation MSE** (lower is better) and
keep the **top 20** as our working population. `summarize` renders that population as text —
this is what the refinement prompt will feed back to the LLM.


In [ ]:
def summarize(results, top_n=20):
    s = '\nTop Model Generation Results:\n'
    for r in results[:top_n]:
        s += f"Model Code:\n{r['model_code']}\n"
        s += f"Validation MSE:\n{r['validation_mse']}\n"
        s += f"Validation R2: {r['validation_r2']}\n\n"
    return s

def phase1(retrieve, data, max_iter=None, log=print):
    max_iter = mcfg['phase1_iterations'] if max_iter is None else max_iter
    results, iteration = [], 0
    while iteration <= max_iter:
        context = retrieve()
        try:
            code_str = clean_code(generate_initial(context))
            log(f'[Phase 1] iter {iteration + 1}: candidate:\n{code_str}')
            func = extract_function(code_str)
            _, mse, r2 = fit_and_evaluate(func, data)
            log(f'[Phase 1] validation R2 = {r2:.4f}')
            iteration += 1
            results.append({'model_code': code_str, 'validation_mse': mse, 'validation_r2': r2})
        except Exception as e:
            log(f'[Phase 1] iter {iteration + 1} failed (skipped): {e}')
            continue
    results.sort(key=lambda x: x['validation_mse'])
    return summarize(results[:20]), results[:20]


Run Phase 1. **Tip:** set `max_iter` to a small number (e.g. `3`) for a quick first run —
the full count in the config can take a while and cost API calls.


In [ ]:
summary_p1, results_p1 = phase1(retrieve, data, max_iter=3)   # raise/remove max_iter for a full run
print('\nBest Phase 1 equation:\n')
print(results_p1[0]['model_code'])
print('validation R2 =', results_p1[0]['validation_r2'])


## 6. Testing — does it *extrapolate*?

Validation scores guided the search, but the real question is performance on the held-out
high-value region. `evaluate_on_test` refits each top model on the training data and scores
it on the **test** set. We report the best model's test MSE/R2 and a full breakdown.

A model that scores well on validation **and** test is one whose functional form genuinely
captures the process physics, not just the training cloud.


In [ ]:
def evaluate_on_test(results, data, top_n=20):
    s = '\nTop Model Test Results:\n'
    best_mse = best_r2 = None
    X_test, y_test = data['test']
    for i in range(min(top_n, len(results))):
        code_str = clean_code(results[i]['model_code'])
        try:
            func = extract_function(code_str)
            n_params = count_inputs(func) - 1
            popt, _ = curve_fit(func, data['train'][0], data['train'][1], p0=[1] * n_params)
            mse = mean_squared_error(y_test, func(X_test, *popt))
            r2  = r2_score(y_test, func(X_test, *popt))
        except Exception as e:
            s += f"Model Code:\n{results[i]['model_code']}\nTest evaluation failed: {e}\n\n"
            continue
        if best_mse is None:
            best_mse, best_r2 = mse, r2
        s += f"Model Code:\n{results[i]['model_code']}\nTest MSE:\n{mse}\nTest R2: {r2}\n\n"
    return best_mse, best_r2, s

init_mse, init_r2, _ = evaluate_on_test(results_p1, data)
print('Initial best -> test MSE:', init_mse, '| test R2:', init_r2)


## 7. Phase 3 — refine until it is good enough

Now we close the loop. Each **refinement round** asks the LLM (via `refine_prompt`) to
improve on the current top-20 summary, generating `phase3_iterations` new candidates which
are fitted, scored and merged back into the population. We re-sort and keep the new top 20.

We keep doing rounds **while the best validation R2 is still more than `r2_threshold` away
from a perfect 1.0**, up to `refinement_max_rounds`. In other words: stop early once the fit
is good enough, otherwise give up after the budget. This is a guided hill-climb where the
LLM proposes the moves and the data picks the winners.


In [ ]:
def refine_once(retrieve, data, summary, results, max_iter=None, log=print):
    max_iter = mcfg['phase3_iterations'] if max_iter is None else max_iter
    iteration = 0
    while iteration <= max_iter:
        try:
            context = retrieve()
            code_str = clean_code(generate_refined(context, summary))
            log(f'[Phase 3] iter {iteration + 1}: candidate:\n{code_str}')
            func = extract_function(code_str)
            _, mse, r2 = fit_and_evaluate(func, data)
            log(f'[Phase 3] validation R2 = {r2:.4f} | MSE = {mse}')
            iteration += 1
            results.append({'model_code': code_str, 'validation_mse': mse, 'validation_r2': r2})
        except Exception as e:
            log(f'[Phase 3] iter {iteration + 1} failed (skipped): {e}')
            continue
    results.sort(key=lambda x: x['validation_mse'])
    return summarize(results[:20]), results[:20]

def phase3(retrieve, data, summary, results, rounds_iter=None, log=print):
    threshold, max_rounds = mcfg['r2_threshold'], mcfg['refinement_max_rounds']
    summary_p3, results_p3, rounds = summary, results, 0
    while (1 - results_p3[0]['validation_r2'] > threshold) and rounds < max_rounds:
        log(f"[Phase 3] round {rounds + 1} (best val R2 = {results_p3[0]['validation_r2']:.4f})")
        summary_p3, results_p3 = refine_once(retrieve, data, summary_p3, results_p3, rounds_iter, log)
        rounds += 1
    return summary_p3, results_p3


In [ ]:
summary_p3, results_p3 = phase3(retrieve, data, summary_p1, list(results_p1), rounds_iter=3)
print('\nBest final equation:\n')
print(results_p3[0]['model_code'])
print('validation R2 =', results_p3[0]['validation_r2'])

final_mse, final_r2, test_summary = evaluate_on_test(results_p3, data)
print('Final best -> test MSE:', final_mse, '| test R2:', final_r2)


## 8. Write the outputs

Finally we persist the results the way `main.py` does: the best **initial** and **final**
equations (each with its validation and test metrics) plus a full run summary. These are the
deliverables a reader of the project cares about.


In [ ]:
out_dir = resolve_path(cfg, cfg['output']['dir'])
out_dir.mkdir(parents=True, exist_ok=True)

def write_equation_file(path, title, best, test_mse, test_r2):
    with open(path, 'w', encoding='utf-8') as f:
        f.write(f'# {title}\n\n')
        f.write(best['model_code'].strip() + '\n\n')
        f.write(f"Validation MSE: {best['validation_mse']}\n")
        f.write(f"Validation R^2: {best['validation_r2']}\n")
        if test_mse is not None:
            f.write(f'Test MSE: {test_mse}\nTest R^2: {test_r2}\n')

write_equation_file(out_dir / cfg['output']['initial_equation_file'],
                    'Initial best equation (Phase 1)', results_p1[0], init_mse, init_r2)
write_equation_file(out_dir / cfg['output']['final_equation_file'],
                    'Final best equation (after refinement)', results_p3[0], final_mse, final_r2)

with open(out_dir / cfg['output']['summary_file'], 'w', encoding='utf-8') as f:
    f.write(f"Topic: {cfg['topic']}\n\n")
    f.write('==== PHASE 1 (INITIAL) ====\n' + summary_p1 + '\n')
    f.write('==== PHASE 3 (REFINED) ====\n' + summary_p3 + '\n')
    f.write('==== TEST RESULTS ====\n' + test_summary + '\n')

print('Wrote outputs to', out_dir)


## Recap

You just ran, with every step visible:

1. **RAG knowledge extraction** — PDFs to a grounded statement of variable relationships.
2. **Extrapolative split** — train on the low-value corner, test on the unseen high-value region.
3. **Phase 1** — sample a population of candidate equations and fit/score them.
4. **Testing** — check extrapolation on the held-out set.
5. **Phase 3** — LLM-guided refinement until the validation R2 target is met.
6. **Outputs** — best initial/final equations + metrics.

To productionise: this is exactly what `python main.py` runs via the `src/` package. To
adapt to a new process, edit `config/config.yaml` only — the logic above is process-agnostic.
